# Lab 5: Playground access to tree canopy

**Question:** Which Montréal playgrounds are surrounded by more tree canopy?

The playgrounds remain the main subject. For each playground, this notebook measures the share of its 250 m circular buffer covered by the existing Ville de Montréal Canopée 2019 polygons. It reads the canopy source in place; it does not download or duplicate it.

**Source CRS note:** The source file is `data/canopee-2019.geojson`. GeoJSON readers report EPSG:4326 by default, but this file's coordinates are in metre-sized values and its feature metadata identifies MTM zone 8 / NAD83 SCRS. Its coordinates align with EPSG:2950, so the notebook assigns that correct CRS to the canopy geometries instead of incorrectly transforming projected coordinates from 4326. Playground coordinates are EPSG:4326 and are transformed to EPSG:2950.


In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import pyogrio
from shapely import make_valid, union_all

project_dir = next(
    (
        candidate
        for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd() / "A2_Yousif")
        if (candidate / "data" / "playgrounds.geojson").is_file()
    ),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Could not locate data/playgrounds.geojson from the current folder")

playground_path = project_dir / "data" / "playgrounds.geojson"
canopy_path = project_dir / "data" / "canopee-2019.geojson"
if not canopy_path.is_file():
    raise FileNotFoundError(f"Canopée source file was not found: {canopy_path}")

playgrounds = gpd.read_file(playground_path)
canopy_info = pyogrio.read_info(canopy_path)
canopy_metadata = gpd.read_file(
    canopy_path,
    rows=1,
    columns=["projection", "reference"],
)

print("Playgrounds dataset")
print("  CRS:", playgrounds.crs)
print("  Feature count:", len(playgrounds))
print("  Geometry types:", playgrounds.geom_type.value_counts().to_dict())
print("  Bounds:", playgrounds.total_bounds.tolist())

print("\nCanopée 2019 dataset")
print("  Exact file:", canopy_path.name)
print("  GeoJSON reader CRS:", canopy_info["crs"])
print("  Source projection metadata:", canopy_metadata.iloc[0]["projection"])
print("  Source reference metadata:", canopy_metadata.iloc[0]["reference"])
print("  Feature count:", canopy_info["features"])
print("  Geometry type:", canopy_info["geometry_type"])
print("  Raw coordinate bounds:", list(canopy_info["total_bounds"]))

assert len(playgrounds) == 1008, f"Expected 1,008 playgrounds, found {len(playgrounds)}"
assert set(playgrounds.geom_type) == {"Point"}
assert playgrounds.crs.to_epsg() == 4326
assert canopy_info["features"] == 514124
assert "fuseau 8" in canopy_metadata.iloc[0]["projection"].lower()
assert "nad83" in canopy_metadata.iloc[0]["reference"].lower()
assert canopy_info["total_bounds"][0] > 180

playgrounds_2950 = playgrounds.to_crs("EPSG:2950")
print("\nPlayground analysis CRS:", playgrounds_2950.crs)


Playgrounds dataset
  CRS: EPSG:4326
  Feature count: 1008
  Geometry types: {'Point': 1008}
  Bounds: [-73.9066169175, 45.4172049709, -73.4783589911, 45.7021020591]

Canopée 2019 dataset
  Exact file: canopee-2019.geojson
  GeoJSON reader CRS: EPSG:4326
  Source projection metadata: Mercator transverse modifiée, fuseau 8
  Source reference metadata: NAD83 SCRS
  Feature count: 514124
  Geometry type: MultiPolygon
  Raw coordinate bounds: [267515.3750000001, 5029228.62499946, 306658.12500000035, 5062653.874999461]

Playground analysis CRS: EPSG:2950


## Create buffers and read only canopy features in the study extent

Each playground gets a 250 m buffer in EPSG:2950. The canopy read covers the snapped 1 km grid extent around the playgrounds. The bbox uses the source's stored projected coordinates; the loaded canopy geometries are then correctly assigned EPSG:2950 based on the source metadata.


In [2]:
playgrounds_2950 = playgrounds_2950.reset_index(drop=True).copy()
playgrounds_2950["playground_id"] = range(len(playgrounds_2950))
buffers = gpd.GeoDataFrame(
    playgrounds_2950.drop(columns="geometry"),
    geometry=playgrounds_2950.geometry.buffer(250),
    crs="EPSG:2950",
)

cell_size = 1000
min_x, min_y, max_x, max_y = playgrounds_2950.total_bounds
grid_origin_x = (min_x // cell_size) * cell_size
grid_origin_y = (min_y // cell_size) * cell_size
grid_columns = int((max_x - grid_origin_x) // cell_size) + 1
grid_rows = int((max_y - grid_origin_y) // cell_size) + 1
canopy_grid_bounds = (
    grid_origin_x,
    grid_origin_y,
    grid_origin_x + grid_columns * cell_size,
    grid_origin_y + grid_rows * cell_size,
)
canopy_window = gpd.read_file(canopy_path, bbox=canopy_grid_bounds, columns=[])
if canopy_window.empty:
    raise ValueError("No Canopée polygons intersect the playground buffer extent")
canopy_2950 = canopy_window.set_crs("EPSG:2950", allow_override=True)

print("Buffer CRS:", buffers.crs)
print("Buffer distance (m):", 250)
print("Buffer count:", len(buffers))
print("Buffer area (m²):", round(float(buffers.geometry.iloc[0].area), 2))
print("Canopy features in buffer extent:", len(canopy_2950))
print("Canopy analysis CRS:", canopy_2950.crs)
invalid_canopy = ~canopy_2950.geometry.is_valid
print("Invalid canopy geometries before repair:", int(invalid_canopy.sum()))
if invalid_canopy.any():
    canopy_2950.loc[invalid_canopy, canopy_2950.geometry.name] = make_valid(
        canopy_2950.loc[invalid_canopy].geometry.array
    )
print("Canopy geometries valid after repair:", bool(canopy_2950.geometry.is_valid.all()))

assert len(buffers) == 1008
assert buffers.crs.to_epsg() == 2950
assert canopy_2950.crs.to_epsg() == 2950
assert canopy_2950.geometry.is_valid.all(), "Canopy geometry repair did not produce valid geometries"


Buffer CRS: EPSG:2950
Buffer distance (m): 250
Buffer count: 1008
Buffer area (m²): 196034.28
Canopy features in buffer extent: 493468
Canopy analysis CRS: EPSG:2950


Invalid canopy geometries before repair: 10


Canopy geometries valid after repair: True


## Measure canopy area and percentage

For each buffer, intersect the candidate canopy polygons, combine the clipped pieces, and measure their union. Combining the pieces prevents overlapping canopy features from being counted twice. `canopy_pct_250m` is the canopy area divided by that playground's full 250 m buffer area, multiplied by 100.


In [3]:
canopy_index = canopy_2950.sindex
canopy_area_m2 = []

for buffer_geometry in buffers.geometry:
    candidate_positions = canopy_index.query(buffer_geometry, predicate="intersects")
    clipped_canopy = [
        canopy_2950.geometry.iloc[position].intersection(buffer_geometry)
        for position in candidate_positions
    ]
    canopy_area_m2.append(float(union_all(clipped_canopy).area) if clipped_canopy else 0.0)

results = playgrounds_2950.copy()
results["buffer_area_250m_m2"] = buffers.geometry.area.to_numpy()
results["canopy_area_250m_m2"] = canopy_area_m2
results["canopy_pct_250m"] = (
    results["canopy_area_250m_m2"] / results["buffer_area_250m_m2"] * 100
)

assert len(results) == 1008
assert results["canopy_pct_250m"].notna().all()
assert (results["canopy_area_250m_m2"] >= 0).all()
assert (results["canopy_pct_250m"].between(0, 100 + 1e-9)).all()

print("Playgrounds analyzed:", len(results))
print("\nCanopy percentage distribution within 250 m")
print(results["canopy_pct_250m"].describe().round(2))

display_columns = ["ID", "OBJECTID", "INDEX_PARC", "ARROND", "canopy_area_250m_m2", "canopy_pct_250m"]
print("\n10 playgrounds with the highest canopy percentage within 250 m")
display(results.nlargest(10, "canopy_pct_250m")[display_columns].round({
    "canopy_area_250m_m2": 1,
    "canopy_pct_250m": 2,
}))

print("\n10 playgrounds with the lowest canopy percentage within 250 m")
display(results.nsmallest(10, "canopy_pct_250m")[display_columns].round({
    "canopy_area_250m_m2": 1,
    "canopy_pct_250m": 2,
}))


Playgrounds analyzed: 1008

Canopy percentage distribution within 250 m
count    1008.00
mean       25.50
std         9.25
min         1.75
25%        19.66
50%        24.48
75%        30.79
max        73.99
Name: canopy_pct_250m, dtype: float64

10 playgrounds with the highest canopy percentage within 250 m


,ID,OBJECTID,INDEX_PARC,ARROND,canopy_area_250m_m2,canopy_pct_250m
30,003-0514,222,0341-000,Le Sud-Ouest,145050.8,73.99
44,003-0262,320,2356-000,L'Île-Bizard-Sainte-Geneviève,137327.7,70.05
433,003-0056,1617,0061-000,Le Plateau-Mont-Royal,111932.7,57.10
351,003-0187,1353,0225-000,Ville-Marie,110204.5,56.22
423,003-0304,1597,2900-000,Verdun,108686.7,55.44
971,003-0256,3389,,L'Île-Bizard-Sainte-Geneviève,108314.7,55.25
886,004-0162,3101,0225-000,Ville-Marie,107872.9,55.03
515,004-0266,1856,2904-000,Verdun,105531.2,53.83
329,003-0424,1282,0921-000,Rivière-des-Prairies-Pointe-aux-Trembles,105374.1,53.75
943,004-0049,3290,0061-000,Le Plateau-Mont-Royal,104716.8,53.42



10 playgrounds with the lowest canopy percentage within 250 m


,ID,OBJECTID,INDEX_PARC,ARROND,canopy_area_250m_m2,canopy_pct_250m
24,004-0458,196,2831-000,Saint-Laurent,3438.2,1.75
993,003-0036,3440,3792-000,Saint-Laurent,7422.5,3.79
404,003-0032,1527,3791-000,Saint-Laurent,7710.0,3.93
753,004-0029,2619,3791-000,Saint-Laurent,8437.4,4.30
15,004-0456,137,3816-000,Saint-Laurent,9008.1,4.60
13,003-0512,126,3816-000,Saint-Laurent,9023.0,4.60
502,003-0507,1800,1293-000,Le Sud-Ouest,9058.9,4.62
827,004-0451,2901,1293-000,Le Sud-Ouest,9882.6,5.04
599,005-0040,2117,2871-014,Saint-Léonard,10500.3,5.36
36,003-0517,258,NaN,Ville-Marie,12181.1,6.21


## Export playgrounds with canopy percentages

Return the enriched playground points to EPSG:4326 for use in Leaflet. Save the derived file beside the source data, then read it back and check its feature count, geometry, CRS, and canopy percentage values.


In [4]:
playgrounds_canopy = results.to_crs("EPSG:4326").copy()
playgrounds_canopy["canopy_pct_250m"] = results["canopy_pct_250m"].to_numpy()
output_path = project_dir / "data" / "playgrounds_canopy.geojson"

assert len(playgrounds_canopy) == 1008
assert playgrounds_canopy.crs.to_epsg() == 4326
assert set(playgrounds_canopy.geom_type) == {"Point"}
assert playgrounds_canopy["canopy_pct_250m"].notna().all()
assert playgrounds_canopy["canopy_pct_250m"].between(0, 100 + 1e-9).all()

playgrounds_canopy.to_file(output_path, driver="GeoJSON")
assert output_path.is_file(), f"Output file was not created: {output_path}"

written_canopy = gpd.read_file(output_path)
assert len(written_canopy) == 1008
assert "canopy_pct_250m" in written_canopy.columns
assert written_canopy["canopy_pct_250m"].notna().all()
assert written_canopy["canopy_pct_250m"].between(0, 100 + 1e-9).all()
assert written_canopy.crs.to_epsg() == 4326
assert set(written_canopy.geom_type) == {"Point"}

print(f"Wrote: {output_path}")
print(f"Playground features: {len(written_canopy)}")
print(f"Canopy percentage range: {written_canopy['canopy_pct_250m'].min():.2f}% to {written_canopy['canopy_pct_250m'].max():.2f}%")
print("Canopy output checks passed")


Wrote: C:\Users\alies\Desktop\GEOG464\A2_Yousif\A2_Yousif\data\playgrounds_canopy.geojson
Playground features: 1008
Canopy percentage range: 1.75% to 73.99%
Canopy output checks passed


## Build a 1 km canopy percentage overlay

Create 1 km squares covering the playground extent. For each square, measure the union of its intersections with the Canopée polygons, divide by the square area, and save a compact EPSG:4326 GeoJSON for the web map.


In [5]:
from shapely.geometry import box

grid_records = []
for row in range(grid_rows):
    for column in range(grid_columns):
        cell_id = row * grid_columns + column
        left = grid_origin_x + column * cell_size
        bottom = grid_origin_y + row * cell_size
        grid_records.append({
            "cell_id": cell_id,
            "geometry": box(left, bottom, left + cell_size, bottom + cell_size),
        })

canopy_grid_2950 = gpd.GeoDataFrame(
    grid_records,
    geometry="geometry",
    crs="EPSG:2950",
)
canopy_index = canopy_2950.sindex
canopy_percentages = []

for cell_geometry in canopy_grid_2950.geometry:
    candidate_positions = canopy_index.query(cell_geometry, predicate="intersects")
    clipped_canopy = [
        canopy_2950.geometry.iloc[position].intersection(cell_geometry)
        for position in candidate_positions
    ]
    canopy_area = float(union_all(clipped_canopy).area) if clipped_canopy else 0.0
    canopy_percentages.append(canopy_area / cell_geometry.area * 100)

canopy_grid_2950["canopy_pct"] = canopy_percentages
canopy_grid = canopy_grid_2950.to_crs("EPSG:4326")
grid_output_path = project_dir / "data" / "canopy_grid.geojson"

assert len(canopy_grid) == grid_rows * grid_columns
assert canopy_grid.crs.to_epsg() == 4326
assert set(canopy_grid.geom_type) == {"Polygon"}
assert pd.api.types.is_numeric_dtype(canopy_grid["canopy_pct"])
assert canopy_grid["canopy_pct"].notna().all()
assert canopy_grid["canopy_pct"].between(0, 100 + 1e-9).all()

canopy_grid.to_file(grid_output_path, driver="GeoJSON")
assert grid_output_path.is_file(), f"Canopy grid file was not created: {grid_output_path}"

written_grid = gpd.read_file(grid_output_path)
assert len(written_grid) == grid_rows * grid_columns
assert "canopy_pct" in written_grid.columns
assert pd.api.types.is_numeric_dtype(written_grid["canopy_pct"])
assert written_grid["canopy_pct"].notna().all()
assert written_grid["canopy_pct"].between(0, 100 + 1e-9).all()
assert written_grid.crs.to_epsg() == 4326
assert set(written_grid.geom_type) == {"Polygon"}

print(f"Canopy grid cells: {len(written_grid)} ({cell_size} m squares)")
print(f"Canopy coverage range: {written_grid['canopy_pct'].min():.2f}% to {written_grid['canopy_pct'].max():.2f}%")
print(f"Wrote: {grid_output_path}")
print("Canopy grid written successfully")
print("All canopy grid checks passed")


Canopy grid cells: 1122 (1000 m squares)
Canopy coverage range: 0.00% to 79.47%
Wrote: C:\Users\alies\Desktop\GEOG464\A2_Yousif\A2_Yousif\data\canopy_grid.geojson
Canopy grid output checks passed


In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

project_dir = next(
    candidate
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd() / "A2_Yousif")
    if (candidate / "data" / "canopy_grid.geojson").is_file()
)
grid_output_path = project_dir / "data" / "canopy_grid.geojson"
grid = gpd.read_file(grid_output_path)
grid_metric = grid.to_crs("EPSG:2950")
grid["cell_area_m2"] = grid_metric.geometry.area.to_numpy()
grid["canopy_area_m2"] = grid["canopy_pct"] / 100 * grid["cell_area_m2"]

final_grid = grid.to_crs("EPSG:4326")
assert grid_output_path.is_file()
assert set(final_grid.geom_type).issubset({"Polygon", "MultiPolygon"})
assert pd.api.types.is_numeric_dtype(final_grid["canopy_pct"])
assert pd.api.types.is_numeric_dtype(final_grid["cell_area_m2"])
assert pd.api.types.is_numeric_dtype(final_grid["canopy_area_m2"])
assert final_grid["canopy_pct"].between(0, 100).all()
assert final_grid["cell_area_m2"].gt(0).all()
assert final_grid["canopy_area_m2"].between(0, final_grid["cell_area_m2"]).all()
assert final_grid.crs.to_epsg() == 4326

final_grid.to_file(grid_output_path, driver="GeoJSON")
assert grid_output_path.is_file()
written_grid = gpd.read_file(grid_output_path)
assert len(written_grid) == len(final_grid)
assert set(written_grid.geom_type).issubset({"Polygon", "MultiPolygon"})
assert pd.api.types.is_numeric_dtype(written_grid["canopy_pct"])
assert pd.api.types.is_numeric_dtype(written_grid["cell_area_m2"])
assert pd.api.types.is_numeric_dtype(written_grid["canopy_area_m2"])
assert written_grid["canopy_pct"].between(0, 100).all()
assert written_grid["cell_area_m2"].gt(0).all()
assert written_grid["canopy_area_m2"].between(0, written_grid["cell_area_m2"]).all()
assert written_grid.crs.to_epsg() == 4326

print('grid["canopy_pct"].describe():')
print(written_grid["canopy_pct"].describe().round(2))

class_counts = pd.Series({
    "0%": int(written_grid["canopy_pct"].eq(0).sum()),
    ">0–<10%": int((written_grid["canopy_pct"].gt(0) & written_grid["canopy_pct"].lt(10)).sum()),
    "10–<20%": int((written_grid["canopy_pct"].ge(10) & written_grid["canopy_pct"].lt(20)).sum()),
    "20–<40%": int((written_grid["canopy_pct"].ge(20) & written_grid["canopy_pct"].lt(40)).sum()),
    "40%+": int(written_grid["canopy_pct"].ge(40).sum()),
})
print("\nCell counts in the proposed map classes:")
print(class_counts.to_string())
assert int(class_counts.sum()) == len(written_grid)
print("Canopy grid written successfully")
print("All canopy grid checks passed")


grid["canopy_pct"].describe():
count    1122.00
mean        9.74
std        13.80
min         0.00
25%         0.00
50%         0.00
75%        18.30
max        79.47
Name: canopy_pct, dtype: float64

Cell counts in the proposed map classes:
0%         567
>0–<10%    157
10–<20%    140
20–<40%    222
40%+        36
Canopy grid written successfully
All canopy grid checks passed


### Canopy grid classes

The inspected grid has a median canopy coverage of 0%, a 75th percentile of about 18.30%, and an observed maximum of about 79.47%. The values are strongly concentrated at the low end, so the map uses five readable groups that distinguish cells with no canopy and retain detail across the common low-to-moderate percentages: **0%**, **greater than 0% to less than 10%**, **10% to less than 20%**, **20% to less than 40%**, and **40% or more**. The last class covers the observed high tail.
